# 📖 Notebook 1: Payment Processing Pipeline

In this notebook we build a simplified payment processing pipeline from scratch.  
You'll see every stage a payment goes through — from the merchant creating a **PaymentIntent**, to charging the customer's card, to recording the result.

## Learning Objectives

By the end of this notebook you'll understand:
- The core entities: **Merchant**, **PaymentIntent**, **Transaction**
- The payment lifecycle state machine (`created → processing → succeeded / failed`)
- How the database tracks every state change in an audit log
- Why payments are asynchronous and what that means for your design

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/payment-system
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `payment_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import uuid
import time
import random

# --- connection helpers (same in every notebook) ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "payment_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# quick smoke test
conn = get_db()
r = get_redis()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM merchants")
print(f"✅ Postgres connected – {cur.fetchone()[0]} merchants in DB")
print(f"✅ Redis connected   – ping={r.ping()}")
cur.close()
conn.close()

## 📋 Requirements & Back-of-the-Envelope

### Functional
1. A merchant **creates a PaymentIntent** and submits it for authorization.
2. We **authorize** with the card network, then record success or failure.
3. Retries are **safe** — the same request never charges twice (Notebook 2).
4. Every movement of money is recorded in a **double-entry ledger** (Notebook 3).
5. Merchants can **query** the status of any payment, and we keep a full audit trail.

**Out of scope**: PCI card storage, 3-D Secure, disputes/chargebacks, payouts, FX.

### Non-functional
| Requirement | Target | Why |
|---|---|---|
| Correctness | **Exactly-once effect.** No double charges, no lost charges. | Every other requirement is negotiable; this one is not |
| Ledger integrity | Debits == credits, always, and provably | An unbalanced ledger means money is unaccounted for |
| Durability | A committed payment survives any single failure | You cannot tell a customer their money is "probably" fine |
| Auth latency | p99 < 1 s | Checkout abandonment climbs sharply past a second |
| Availability | 99.99% | ~52 min/year. Downtime is directly lost revenue for every merchant |
| Retention | 7 years, immutable | Financial regulation, not a product choice |

Notice what's *not* here: throughput. A payment system is not a high-QPS
problem — it's a correctness problem that happens to have some traffic. Say
that out loud in an interview and then prove you know the numbers anyway.

In [ ]:
# ── Back-of-the-envelope ────────────────────────────────────────────────
PAYMENTS_PER_DAY   = 100_000_000
PEAK_MULTIPLIER    = 5           # Black Friday, not a normal Tuesday
INTENT_BYTES       = 300
TXN_BYTES          = 300
LEDGER_ROW_BYTES   = 200
AUDIT_ROW_BYTES    = 300
LEDGER_ROWS        = 2           # one debit, one credit per charge
AUDIT_ROWS         = 4           # created / processing / succeeded + txn rows
RETENTION_YEARS    = 7           # regulatory, not optional
SEC_PER_DAY        = 86_400

payments_per_s = PAYMENTS_PER_DAY / SEC_PER_DAY
peak_per_s     = payments_per_s * PEAK_MULTIPLIER

bytes_per_payment = (INTENT_BYTES + TXN_BYTES
                     + LEDGER_ROWS * LEDGER_ROW_BYTES
                     + AUDIT_ROWS * AUDIT_ROW_BYTES)
bytes_per_day  = PAYMENTS_PER_DAY * bytes_per_payment
bytes_retained = bytes_per_day * 365 * RETENTION_YEARS

ledger_rows_per_day = PAYMENTS_PER_DAY * LEDGER_ROWS
ledger_rows_total   = ledger_rows_per_day * 365 * RETENTION_YEARS

# How long does a naive full-ledger balance check take at that size?
ROWS_SCANNED_PER_SEC = 5_000_000   # generous for a big sequential aggregate
full_scan_seconds = ledger_rows_total / ROWS_SCANNED_PER_SEC

print("📐 Back-of-the-envelope")
print("=" * 70)
print(f"  Payments:            {payments_per_s:>14,.0f} /s avg   "
      f"{peak_per_s:>10,.0f} /s peak")
print(f"  Bytes per payment:   {bytes_per_payment:>14,} B "
      f"(intent + txn + {LEDGER_ROWS} ledger + {AUDIT_ROWS} audit)")
print(f"  Storage:             {bytes_per_day / 1e9:>14,.0f} GB/day   "
      f"{bytes_per_day * 365 / 1e12:>7,.0f} TB/year")
print(f"  Retained {RETENTION_YEARS} years:   {bytes_retained / 1e12:>14,.0f} TB")
print()
print(f"  Ledger rows/day:     {ledger_rows_per_day:>14,.0f} "
      f"({ledger_rows_per_day / SEC_PER_DAY:,.0f} writes/s)")
print(f"  Ledger rows total:   {ledger_rows_total / 1e12:>14,.2f} trillion")
print()
print("  ⚠️  Naive whole-ledger balance check "
      "(SUM over every row ever written):")
print(f"      {full_scan_seconds / 3600:>10,.1f} hours per run  ❌")
print("      Notebook 3's check_books_balance() does exactly this. It is")
print("      correct and it does not scale — see the note there.")

### What those numbers decide

- **~1,160 payments/s average, ~5,800/s at peak.** By web standards that is
  *small*. One well-tuned Postgres primary handles it. Resist the urge to
  design a distributed system for the throughput; design it for the correctness.
- **~1.8 KB written per payment across four tables.** The intent, the
  transaction, the two ledger rows and the audit rows must land atomically or
  you have a reconciliation problem instead of a payment. That argues strongly
  for keeping them in **one database, one transaction** for as long as you
  possibly can. Splitting the ledger into its own service is the single
  most expensive architectural decision in this design — it turns a `COMMIT`
  into a distributed saga.
- **~64 TB/year, ~450 TB retained.** Almost all of it is cold. Partition by
  month; only the current partition needs to be fast.
- **~1.4 trillion ledger rows over the retention window.** This is the number
  that breaks the obvious implementation: you cannot verify the books by
  summing the whole ledger. Real systems **close the books periodically** —
  compute and store a signed daily balance per account, then verify only the
  current day against yesterday's closing balance. Verification becomes O(one
  day) instead of O(all history).

**The trade-off in that last point:** closing balances are a *derived* value you
now have to keep correct, and a bug in the closing job silently corrupts the
baseline for every future check. The mitigation is to keep the full scan as a
monthly (or quarterly) job against a read replica — expensive, slow, and the
only thing that can catch a bad baseline.

---
## 1. Exploring the Core Entities

Our payment system has three main entities:

| Entity | Purpose |
|--------|---------|
| **Merchant** | A business using our platform (has an API key) |
| **PaymentIntent** | The merchant's *intention* to collect a specific amount |
| **Transaction** | A single money-movement attempt linked to a PaymentIntent |

One PaymentIntent can have **many** Transactions.  
Why? If the first charge attempt fails (e.g. insufficient funds), the merchant retries, creating a new Transaction under the same PaymentIntent.

Let's look at what's already seeded in our database.

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Show all merchants
cur.execute("SELECT id, name, email FROM merchants")
print("=== Merchants ===")
for row in cur.fetchall():
    print(f"  {row['id']}  {row['name']}  ({row['email']})")

# Show payment intent counts by status
cur.execute("""
    SELECT status, COUNT(*) as cnt
    FROM payment_intents
    GROUP BY status
    ORDER BY cnt DESC
""")
print("\n=== PaymentIntent Status Summary ===")
for row in cur.fetchall():
    print(f"  {row['status']:15s}  {row['cnt']}")

cur.close()
conn.close()

---
## 2. The Payment Lifecycle – Step by Step

A payment goes through a simple state machine:

```
  created  ──►  processing  ──►  succeeded
                            ──►  failed
```

1. **Created** – The merchant tells us "I want to charge customer X for $49.99."
2. **Processing** – We send the charge to the card network (Visa, Mastercard, etc.).
3. **Succeeded / Failed** – The network responds with approval or decline.

Let's implement this pipeline as Python functions.

In [ ]:
def create_payment_intent(merchant_id, amount_cents, currency, description, idempotency_key=None):
    """
    Step 1: Merchant creates a PaymentIntent.
    This is like telling us 'I intend to charge this customer $X'.
    No money moves yet — we just record the intention.
    """
    pi_id = f"pi_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status, idempotency_key)
            VALUES (%s, %s, %s, %s, %s, 'created', %s)
            RETURNING id, status
        """, (pi_id, merchant_id, amount_cents, currency, description, idempotency_key))

        result = cur.fetchone()

        # Record the state change in our audit log
        cur.execute("""
            INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by, metadata)
            VALUES ('payment_intent', %s, NULL, 'created', 'payment_service', %s)
        """, (pi_id, json.dumps({"amount_cents": amount_cents, "merchant_id": merchant_id})))

        conn.commit()
        print(f"✅ PaymentIntent created: {result[0]}  status={result[1]}")
        return result[0]
    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        # Idempotency key already used — return existing intent
        cur.execute("""
            SELECT id, status FROM payment_intents
            WHERE merchant_id = %s AND idempotency_key = %s
        """, (merchant_id, idempotency_key))
        existing = cur.fetchone()
        print(f"⚠️  Idempotency key already used — returning existing: {existing[0]}  status={existing[1]}")
        return existing[0]
    finally:
        cur.close()
        conn.close()

# Try it!
pi_id = create_payment_intent("merch_001", 3500, "usd", "Order #2001 - Notebook demo")
print(f"   Returned PaymentIntent ID: {pi_id}")

In [ ]:
def simulate_card_network(card_last_four):
    """
    Simulates calling an external card network (Visa, Mastercard, etc.).
    In reality this would be an HTTP call to a payment network over a
    private, PCI-compliant connection.  Here we just fake it:
      - card ending '0000' always fails (insufficient funds)
      - card ending '9999' always times out
      - everything else succeeds after a short delay
    """
    time.sleep(random.uniform(0.1, 0.5))  # simulate network latency

    if card_last_four == "0000":
        return {"status": "declined", "reason": "insufficient_funds", "network_ref": None}
    if card_last_four == "9999":
        raise TimeoutError("Card network did not respond in time")

    return {
        "status": "approved",
        "reason": None,
        "network_ref": f"net_{uuid.uuid4().hex[:8]}"
    }

print("Card network simulator ready.")
print("  '4242' → approved")
print("  '0000' → declined (insufficient funds)")
print("  '9999' → timeout")

In [ ]:
def process_payment(payment_intent_id, card_last_four, card_brand="visa"):
    """
    Step 2 & 3: Create a Transaction, call the card network, update statuses.

    This is where money actually moves (or fails to move).
    """
    txn_id = f"txn_{uuid.uuid4().hex[:12]}"
    conn = get_db()
    cur = conn.cursor()

    try:
        # --- Fetch the PaymentIntent so we know the amount ---
        cur.execute("SELECT amount_cents, currency, status FROM payment_intents WHERE id = %s", (payment_intent_id,))
        pi = cur.fetchone()
        if pi is None:
            print(f"❌ PaymentIntent {payment_intent_id} not found"); return None

        amount_cents, currency, pi_status = pi
        if pi_status not in ('created', 'failed'):
            print(f"❌ PaymentIntent is '{pi_status}' — cannot process again"); return None

        # --- Move PaymentIntent to 'processing' ---
        cur.execute("UPDATE payment_intents SET status='processing', updated_at=NOW() WHERE id=%s", (payment_intent_id,))
        cur.execute("""
            INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by)
            VALUES ('payment_intent', %s, %s, 'processing', 'payment_service')
        """, (payment_intent_id, pi_status))

        # --- Create the Transaction (pending) ---
        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'charge', %s, %s, 'pending', %s, %s)
        """, (txn_id, payment_intent_id, amount_cents, currency, card_last_four, card_brand))
        cur.execute("""
            INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by)
            VALUES ('transaction', %s, NULL, 'pending', 'transaction_service')
        """, (txn_id,))
        conn.commit()
        print(f"⏳ Transaction {txn_id} created (pending) — calling card network...")

        # --- Call the card network (the async, unreliable outside world) ---
        try:
            result = simulate_card_network(card_last_four)
        except TimeoutError as e:
            # Timeout: we don't know if the charge went through!
            cur.execute("UPDATE transactions SET status='timeout', updated_at=NOW() WHERE id=%s", (txn_id,))
            cur.execute("UPDATE payment_intents SET status='processing', updated_at=NOW() WHERE id=%s", (payment_intent_id,))
            cur.execute("""
                INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by, metadata)
                VALUES ('transaction', %s, 'pending', 'timeout', 'transaction_service', %s)
            """, (txn_id, json.dumps({"error": str(e)})))
            conn.commit()
            print(f"⏱️  Timeout!  Transaction {txn_id} → status=timeout")
            print("   In production a reconciliation service would check with the network later.")
            return txn_id

        # --- Handle the response ---
        if result["status"] == "approved":
            cur.execute("""
                UPDATE transactions SET status='succeeded', network_reference_id=%s, updated_at=NOW() WHERE id=%s
            """, (result["network_ref"], txn_id))
            cur.execute("UPDATE payment_intents SET status='succeeded', updated_at=NOW() WHERE id=%s", (payment_intent_id,))
            new_status = "succeeded"
        else:
            cur.execute("""
                UPDATE transactions SET status='failed', failure_reason=%s, updated_at=NOW() WHERE id=%s
            """, (result["reason"], txn_id))
            cur.execute("UPDATE payment_intents SET status='failed', updated_at=NOW() WHERE id=%s", (payment_intent_id,))
            new_status = "failed"

        # audit log entries
        cur.execute("""
            INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by, metadata)
            VALUES ('transaction', %s, 'pending', %s, 'transaction_service', %s)
        """, (txn_id, new_status, json.dumps(result)))
        cur.execute("""
            INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by)
            VALUES ('payment_intent', %s, 'processing', %s, 'payment_service')
        """, (payment_intent_id, new_status))

        conn.commit()
        emoji = "✅" if new_status == "succeeded" else "❌"
        print(f"{emoji} Transaction {txn_id} → status={new_status}")
        return txn_id
    finally:
        cur.close()
        conn.close()

print("process_payment() ready.")

### 2a. Successful Payment

In [ ]:
# Create a new PaymentIntent and process it with a good card
pi = create_payment_intent("merch_001", 2500, "usd", "Demo: successful charge")
txn = process_payment(pi, card_last_four="4242")

### 2b. Failed Payment (Insufficient Funds)

In [ ]:
# Create a PaymentIntent and try with a card that always declines
pi_fail = create_payment_intent("merch_002", 9999, "usd", "Demo: declined charge")
txn_fail = process_payment(pi_fail, card_last_four="0000")

### 2c. Timeout (Network Uncertainty)

In [ ]:
# This card always times out — we can't tell if the charge went through!
pi_timeout = create_payment_intent("merch_001", 5000, "usd", "Demo: timeout")
txn_timeout = process_payment(pi_timeout, card_last_four="9999")

---
## 3. Checking the Audit Trail

Every state change is logged in `payment_audit_log`.  
This is critical for payment systems — you must be able to reconstruct exactly what happened with any payment, months or years later.

Let's look at the full audit trail for our successful payment:

In [ ]:
def show_audit_trail(entity_id):
    """Show the complete audit trail for a payment entity."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT entity_type, entity_id, old_status, new_status, changed_by, created_at
        FROM payment_audit_log
        WHERE entity_id = %s
        ORDER BY created_at
    """, (entity_id,))
    rows = cur.fetchall()
    cur.close()
    conn.close()

    if not rows:
        print(f"No audit records found for {entity_id}"); return

    print(f"Audit trail for {entity_id}:")
    for r in rows:
        old = r['old_status'] or '(none)'
        print(f"  {r['created_at']}  {old:15s} → {r['new_status']:15s}  by {r['changed_by']}")

# Show the trail for our successful payment
show_audit_trail(pi)

---
## 4. Caching Payment Status in Redis

Merchants frequently poll `GET /payment-intents/{id}` to check the status of a payment.  
We can avoid hitting the database every time by caching the result in Redis with a short TTL.

This is especially important during checkout — the merchant's frontend might poll every second until the payment succeeds.

In [ ]:
def get_payment_status(payment_intent_id):
    """
    Get the status of a PaymentIntent.
    Checks Redis cache first, falls back to Postgres.
    """
    r = get_redis()
    cache_key = f"pi_status:{payment_intent_id}"

    # Try cache first
    cached = r.get(cache_key)
    if cached:
        print(f"  ⚡ Cache HIT  – {payment_intent_id} → {cached}")
        return cached

    # Cache miss — query Postgres
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT status FROM payment_intents WHERE id = %s", (payment_intent_id,))
    row = cur.fetchone()
    cur.close()
    conn.close()

    if row is None:
        print(f"  ❌ PaymentIntent {payment_intent_id} not found")
        return None

    status = row[0]
    # Cache for 10 seconds (short TTL because status can change quickly)
    r.setex(cache_key, 10, status)
    print(f"  🐘 Cache MISS – queried Postgres – {payment_intent_id} → {status}  (cached for 10s)")
    return status

# First call: cache miss (hits Postgres)
get_payment_status(pi)

# Second call: cache hit (Redis only)
get_payment_status(pi)

---
## 5. Closing the Loop: Reconciliation for Timeouts

Remember the timeout case? The card network never replied, so the PaymentIntent is stuck in `processing` and the Transaction in `timeout`. We *do not know* if the money moved.

Real payment systems run a **reconciliation job** on a schedule: for every transaction stuck in `timeout` or `pending` past a deadline, they query the card network directly (did this ever go through?) and update our state to match theirs. The card network is the source of truth -- our database is just a cached view of what it says.

Below we simulate that job: we ask the (fake) network whether it has a record of each stuck transaction, and close it out one way or the other.


In [ ]:
def simulate_network_lookup(txn_id):
    """
    Pretend we are calling the card network's 'lookup by our reference' endpoint.
    In the real world you would get back the true outcome: approved, declined, or unknown.
    Here we just flip a coin for demo purposes.
    """
    return random.choice([
        {"status": "approved", "network_ref": f"net_recon_{uuid.uuid4().hex[:8]}"},
        {"status": "declined", "reason": "do_not_honor"},
    ])

def reconcile_stuck_transactions(max_age_seconds=0):
    """
    Find transactions stuck in 'timeout' (or 'pending') and resolve them by
    asking the card network for the real outcome.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT id, payment_intent_id, status
        FROM transactions
        WHERE status IN ('timeout', 'pending')
          AND updated_at <= NOW() - (%s || ' seconds')::interval
    """, (max_age_seconds,))
    stuck = cur.fetchall()
    print(f"Found {len(stuck)} stuck transaction(s) to reconcile")
    print()

    for row in stuck:
        txn_id, pi_id, old_status = row['id'], row['payment_intent_id'], row['status']
        outcome = simulate_network_lookup(txn_id)
        print(f"  {txn_id}  (was {old_status}) -> network says: {outcome['status']}")

        if outcome['status'] == 'approved':
            cur.execute("UPDATE transactions SET status='succeeded', network_reference_id=%s, updated_at=NOW() WHERE id=%s",
                        (outcome['network_ref'], txn_id))
            cur.execute("UPDATE payment_intents SET status='succeeded', updated_at=NOW() WHERE id=%s", (pi_id,))
            new_status = 'succeeded'
        else:
            cur.execute("UPDATE transactions SET status='failed', failure_reason=%s, updated_at=NOW() WHERE id=%s",
                        (outcome.get('reason','reconciliation_declined'), txn_id))
            cur.execute("UPDATE payment_intents SET status='failed', updated_at=NOW() WHERE id=%s", (pi_id,))
            new_status = 'failed'

        cur.execute("""
            INSERT INTO payment_audit_log (entity_type, entity_id, old_status, new_status, changed_by, metadata)
            VALUES ('transaction', %s, %s, %s, 'reconciliation_job', %s)
        """, (txn_id, old_status, new_status, json.dumps(outcome)))
    conn.commit()
    cur.close(); conn.close()
    print()
    print("Reconciliation complete -- no transaction is left in an unknown state.")

reconcile_stuck_transactions(max_age_seconds=0)


---
## 6. Summary

Let's see the full picture of what we built:

In [ ]:
conn = get_db()
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM payment_intents")
print(f"Total PaymentIntents : {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM transactions")
print(f"Total Transactions   : {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM payment_audit_log")
print(f"Total Audit Log Rows : {cur.fetchone()[0]}")

cur.close()
conn.close()

print("\n📝 Key takeaways:")
print("  1. A PaymentIntent is the top-level object — it owns the lifecycle.")
print("  2. Transactions are individual charge attempts under a PaymentIntent.")
print("  3. Every state change is recorded in the audit log (append-only).")
print("  4. Timeouts are NOT failures — they require reconciliation.")
print("  5. Redis can cache status for fast polling without hammering Postgres.")
print("\n➡️  Next notebook: Idempotency & Exactly-Once Payments")